In [1]:
from datetime import datetime
import re
import pandas as pd
import spacy

In [9]:


# -------------------------------------------------------------------
# STEP 1: Load spaCy Model & Input CSV
# -------------------------------------------------------------------
print("--> Step 1: Loading spaCy model and reading metadata.csv...")
nlp = spacy.load("en_core_web_sm")

input_csv = "metadata.csv"
output_csv = "histolines_wikiart.csv"

df = pd.read_csv(input_csv)
print(f"Loaded {len(df)} rows from '{input_csv}'.\n")

today_str = datetime.now().strftime("%Y-%m-%d")
processed_rows = []

--> Step 1: Loading spaCy model and reading metadata.csv...
Loaded 215028 rows from 'metadata.csv'.



In [15]:
from datetime import datetime
import re
import pandas as pd
import spacy

# -------------------------------------------------------------------
# STEP 1: Load spaCy Model & Input CSV
# -------------------------------------------------------------------
print("--> Step 1: Loading spaCy model and reading metadata.csv...")
nlp = spacy.load("en_core_web_sm")

input_csv = "metadata.csv"
output_csv = "histolines_wikiart.csv"

df = pd.read_csv(input_csv)
print(f"Loaded {len(df)} rows from '{input_csv}'.\n")

today_str = datetime.now().strftime("%Y-%m-%d")
processed_rows = []

# -------------------------------------------------------------------
# STEP 2: Process line-by-line with step-by-step terminal output
# -------------------------------------------------------------------
print("--> Step 2: Processing rows line-by-line...\n")

for idx, row in df.iterrows():
    event_id = idx + 1

    # Extract raw column values
    caption = (
        str(row["wikiart_caption"])
        if pd.notna(row.get("wikiart_caption"))
        else ""
    )
    file_link = (
        str(row["file_link"]) if pd.notna(row.get("file_link")) else ""
    )
    caption_florence = (
        str(row["CAPTION_Florence-2-large-ft"])
        if pd.notna(row.get("CAPTION_Florence-2-large-ft"))
        else ""
    )

    # --- Step 2a: Parse Artwork Title, Artist Name, and Year from Caption ---
    parts = [p.strip() for p in caption.split(",") if p.strip()]

    title = ""
    artist = ""
    year = ""

    by_idx = -1
    for i, part in enumerate(parts):
        if part.lower().startswith("by "):
            by_idx = i
            artist = re.sub(
                r"^by\s+", "", part, flags=re.IGNORECASE
            ).title()  # Remove 'by ' and format
            break

    if by_idx != -1:
        # Artwork title is everything before "by"
        title = ", ".join(parts[:by_idx])
        rest_parts = parts[by_idx + 1 :]

        # Check if item immediately after artist is a 4-digit year
        if rest_parts and re.match(r"^\d{4}$", rest_parts[0]):
            year = rest_parts[0]
    else:
        # Fallback if "by" is not present
        title = parts[0] if parts else ""

    # --- Step 2b: Run NLP on wikiart_caption for Locations & People Names ---
    doc = nlp(caption)

    # Find the FIRST proper location (GPE = Cities/Countries, FAC = Addresses/Buildings)
    first_location = ""
    for ent in doc.ents:
        if ent.label_ in ["GPE", "FAC"]:
            first_location = ent.text
            break  # Stop as soon as the first proper location is found

    # Find ALL people's names in the caption to store in orgCont
    people_names = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]
    org_cont_names = ", ".join(dict.fromkeys(people_names))

    # --- Step 2c: Map to Target Histolines Columns ---
    record = {
        "eventId": event_id,
        "whoCharName": artist,
        "whatEventType": "took a picture",
        "whenYear": year,
        "whenMonth": "",
        "whenDay": "",
        "whereLat": "",
        "whereLon": "",
        "whereLoc": first_location,
        "whoRel": title,
        "whatRef": caption_florence,
        "enterDate": today_str,
        "eventSig": 5,
        "eventAccYes": "",
        "eventAccNo": "",
        "orgCont": org_cont_names,
        "eventDetail": file_link,
        "access": "public",
    }

    processed_rows.append(record)

    # --- Step 2d: Live Terminal Debug Output ---
    print(f"--- [Row #{event_id}] ---")
    print(f"  whoRel (Artwork Name)       : {title}")
    print(f"  whoCharName (Artist)        : {artist}")
    print(f"  whenYear (Year)             : {year}")
    print(f"  whereLoc (First Location)   : {first_location}")
    print(f"  orgCont (People Found)      : {org_cont_names}")
    print(f"  whatRef (Florence Caption)  : {caption_florence}")
    print(f"  eventDetail (File Link)     : {file_link}")
    print("-" * 60)

Streaming output truncated to the last 5000 lines.
  whereLoc (First Location)   : Italy
  orgCont (People Found)      : 
  whatRef (Florence Caption)  : A painting of a crowd of people outside a building.
  eventDetail (File Link)     : https://uploads5.wikiart.org/00360/images/gerolamo-induno/induno-coscritti.jpg
------------------------------------------------------------
--- [Row #214474] ---
  whoRel (Artwork Name)       : Relational painting, Tondo #4
  whoCharName (Artist)        : Fritz Glarner
  whenYear (Year)             : 1946
  whereLoc (First Location)   : 
  orgCont (People Found)      : fritz glarner
  whatRef (Florence Caption)  : A circle with different colored lines on it.
  eventDetail (File Link)     : https://uploads8.wikiart.org/images/fritz-glarner/relational-painting-tondo-4-1946.jpg
------------------------------------------------------------
--- [Row #214475] ---
  whoRel (Artwork Name)       : View of Genoa
  whoCharName (Artist)        : Francesco Zucchi
  

In [16]:
# -------------------------------------------------------------------
# STEP 2.5: Pre-Export Check & Re-assignment Logic
# -------------------------------------------------------------------
print(
    "\n--> Step 2.5: Running validation on orgCont names vs whoCharName...\n"
)

for record in processed_rows:
    org_cont_val = record["orgCont"]
    artist_val = record["whoCharName"].strip().lower()

    if org_cont_val:
        # Split comma-separated names in orgCont
        names_list = [n.strip() for n in org_cont_val.split(",") if n.strip()]

        # Search for the first name that does NOT match whoCharName
        first_other_name = None
        for name in names_list:
            if name.lower() != artist_val:
                first_other_name = name
                break

        # If a non-artist name is found, perform the move and merge
        if first_other_name:
            original_who_rel = record["whoRel"]

            # Merge whoRel with whatRef
            if original_who_rel:
                if record["whatRef"]:
                    record["whatRef"] = (
                        f"{original_who_rel} - {record['whatRef']}"
                    )
                else:
                    record["whatRef"] = original_who_rel

            # Move non-artist name to whoRel
            record["whoRel"] = first_other_name

            print(
                f"  [Row #{record['eventId']}] MODIFIED: Found non-artist name '{first_other_name}' in orgCont."
            )
            print(f"    --> Moved '{first_other_name}' to whoRel")
            print(
                f"    --> Combined artwork name into whatRef: '{record['whatRef']}'\n"
            )

Streaming output truncated to the last 5000 lines.
  [Row #212770] MODIFIED: Found non-artist name 'Jesus Christ' in orgCont.
    --> Moved 'Jesus Christ' to whoRel
    --> Combined artwork name into whatRef: 'Coronation of the Virgin - A painting of a religious scene with angels and people.'

  [Row #212771] MODIFIED: Found non-artist name 'jean leon' in orgCont.
    --> Moved 'jean leon' to whoRel
    --> Combined artwork name into whatRef: 'A Japanese Imploring a Divinity - A person standing in front of a stairway leading to a large statue.'

  [Row #212773] MODIFIED: Found non-artist name 'Sarah Bernhardt' in orgCont.
    --> Moved 'Sarah Bernhardt' to whoRel
    --> Combined artwork name into whatRef: 'Portrait of Sarah Bernhardt - A painting of a woman in a white dress.'

  [Row #212779] MODIFIED: Found non-artist name 'Vesuvius' in orgCont.
    --> Moved 'Vesuvius' to whoRel
    --> Combined artwork name into whatRef: 'The Bay of Naples at moonlight night. Vesuvius - A painting 

In [18]:
# -------------------------------------------------------------------
# STEP 3: Create DataFrame and Export CSV
# -------------------------------------------------------------------
print("\n--> Step 3: Exporting processed records to CSV...")

columns_order = [
    "eventId",
    "whoCharName",
    "whatEventType",
    "whenYear",
    "whenMonth",
    "whenDay",
    "whereLat",
    "whereLon",
    "whereLoc",
    "whoRel",
    "whatRef",
    "enterDate",
    "eventSig",
    "eventAccYes",
    "eventAccNo",
    "orgCont",
    "eventDetail",
    "access",
]

out_df = pd.DataFrame(processed_rows, columns=columns_order)
out_df.to_csv(output_csv, index=False)

print(f"\nSUCCESS! Created '{output_csv}' with {len(out_df)} rows.")


--> Step 3: Exporting processed records to CSV...


KeyboardInterrupt: 